# EuroSAT EfficientNetB0 Kaggle Runner

This notebook runs Phase 5 EfficientNetB0 Stage 1 and Stage 2 on Kaggle.
It assumes dataset input is mounted under `/kaggle/input` and all outputs are written to `/kaggle/working`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO = WORKING / 'satellite-land-classification-dl'

if REPO.exists():
    shutil.rmtree(REPO)

token = os.environ.get('GH_TOKEN')
if not token:
    raise RuntimeError('ERROR: Miloše, dodaj GH_TOKEN u Kaggle Secrets da bi skripta mogla da povuče kod.')

clone_url = f'https://{token}@github.com/milanovicmilos/satellite-land-classification-dl.git'

!git clone {clone_url} {REPO.as_posix()}
!git -C {REPO.as_posix()} remote set-url origin https://github.com/milanovicmilos/satellite-land-classification-dl.git

checkout = subprocess.run(
    ['git', '-C', REPO.as_posix(), 'checkout', 'feature/efficientnet-optimization'],
    capture_output=True,
    text=True,
)
if checkout.returncode != 0:
    print('WARNING: feature/efficientnet-optimization nije dostupna. Prelazim na main.')
    subprocess.run(['git', '-C', REPO.as_posix(), 'checkout', 'main'], check=True)

%cd {REPO.as_posix()}

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -e .

## Dataset Path Setup

This notebook uses the same optimized EfficientNet configuration files as local execution.
Only `dataset_root` and Stage 2 `resume_from` are patched for the Kaggle runtime paths.

Dataset path is expected at `/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT`.
Training augmentation is controlled by config (`augmentation_mode`) and is set to `flips` in optimized configs.

In [ ]:
import json

DATASET_INPUT_ROOT = Path('/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT')
STAGE1_RESUME_PATH = Path('/kaggle/working/checkpoints/efficientnet_b0/stage1/best_checkpoint.pt')
assert DATASET_INPUT_ROOT.exists(), f'Missing dataset path: {DATASET_INPUT_ROOT}'

CONFIG_STAGE1 = REPO / 'configs' / 'efficientnet_b0.stage1.optimized.json'
CONFIG_STAGE2 = REPO / 'configs' / 'efficientnet_b0.stage2.optimized.json'
DEFAULTS = REPO / 'configs' / 'experiment.defaults.json'

def patch_runtime_paths(config_path: Path) -> None:
    payload = json.loads(config_path.read_text(encoding='utf-8'))
    payload['dataset_root'] = DATASET_INPUT_ROOT.as_posix()

    if 'stage2' in config_path.name:
        payload['training']['resume_from'] = STAGE1_RESUME_PATH.as_posix()

    config_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')

patch_runtime_paths(CONFIG_STAGE1)
patch_runtime_paths(CONFIG_STAGE2)
print('Patched dataset_root:', DATASET_INPUT_ROOT)
print('Patched stage2 resume_from:', STAGE1_RESUME_PATH)

## Prepare Deterministic Splits

In [ ]:
!python run.py --prepare-dataset --config configs/efficientnet_b0.stage1.optimized.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits

## Stage 1 Full Run (Frozen Backbone)

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage1.optimized.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage1_final.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage1

## Stage 2 Full Run (Unfrozen Backbone, Resume from Stage 1)

Stage 2 uses lower learning rate (0.0003) and higher early stopping patience (5) for finer backbone adaptation during full fine-tuning.
The config patch cell rewrites `resume_from` to the absolute Kaggle path `/kaggle/working/checkpoints/efficientnet_b0/stage1/best_checkpoint.pt`.
Run Stage 1 first so the Stage 2 resume checkpoint exists before starting the unfrozen pass.

In [ ]:
!python run.py --run-baseline --config configs/efficientnet_b0.stage2.optimized.json --defaults configs/experiment.defaults.json --splits-output /kaggle/working/artifacts/splits --reports-output /kaggle/working/artifacts/reports/efficientnet_b0_stage2_final.json --checkpoints-output /kaggle/working/checkpoints/efficientnet_b0/stage2

## Export Kaggle Artifacts

This exports reports, checkpoints, and split artifacts produced by the full Stage 1 and Stage 2 runs.

In [ ]:
!zip -r /kaggle/working/results.zip /kaggle/working/artifacts /kaggle/working/checkpoints